In [ ]:
# Kaggle API directory setup
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Amazon Fine Food Reviews dataset download aur extract karna
!kaggle datasets download -d snap/amazon-fine-food-reviews
!unzip -q amazon-fine-food-reviews.zip

# Required libraries install karna
!pip install -q transformers datasets evaluate accelerate

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
100% 242M/242M [00:11<00:00, 21.2MB/s]



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Data load karna
df = pd.read_csv('Reviews.csv')
df = df[['Score', 'Text']].dropna()

# 3-Class mapping function: 1-2 -> 0 (Neg), 3 -> 1 (Neu), 4-5 -> 2 (Pos)
def map_sentiment(rating):
    if rating in [1, 2]:
        return 0  # Negative
    elif rating == 3:
        return 1  # Neutral
    else:
        return 2  # Positive

df['label'] = df['Score'].apply(map_sentiment)

# Class imbalance handle karne ke liye stratified sample (Total 15,000 reviews for fast & stable training)
df_sample = df.groupby('label', group_keys=False).apply(lambda x: x.sample(5000, random_state=42)).reset_index(drop=True)

# Train-Validation-Test Split (70% Train, 15% Val, 15% Test)
train_df, temp_df = train_test_split(df_sample, test_size=0.30, random_state=42, stratify=df_sample['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['label'])

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

Train size: 10500, Val size: 2250, Test size: 2250


/tmp/ipykernel_3451/3782376889.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sample = df.groupby('label', group_keys=False).apply(lambda x: x.sample(5000, random_state=42)).reset_index(drop=True)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

# Simple word tokenizer aur vocab build karna
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

words = [word for text in train_df['Text'] for word in tokenize(text)]
vocab = {word: i + 2 for i, (word, _) in enumerate(Counter(words).most_common(20000))}
vocab['<pad>'] = 0
vocab['<unk>'] = 1

def encode(text, max_len=100):
    tokens = [vocab.get(w, 1) for w in tokenize(text)][:max_len]
    return tokens + [0] * (max_len - len(tokens))

# PyTorch Dataset
class TextDataset(Dataset):
    def __init__(self, df):
        self.texts = [encode(t) for t in df['Text']]
        self.labels = df['label'].values

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.texts[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_loader_lstm = DataLoader(TextDataset(train_df), batch_size=64, shuffle=True)
test_loader_lstm = DataLoader(TextDataset(test_df), batch_size=64)

# Bi-LSTM Architecture
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        # Bi-directional ke last hidden states concatenate karna
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        return self.fc(self.dropout(hidden))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lstm_model = SentimentLSTM(len(vocab) + 2, 128, 64, 3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

# LSTM Training Loop
print("Training LSTM Baseline...")
lstm_model.train()
for epoch in range(4):
    total_loss = 0
    for texts, labels in train_loader_lstm:
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(lstm_model(texts), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/4 | Loss: {total_loss/len(train_loader_lstm):.4f}")

Training LSTM Baseline...
Epoch 1/4 | Loss: 1.0843
Epoch 2/4 | Loss: 1.0290
Epoch 3/4 | Loss: 0.9603
Epoch 4/4 | Loss: 0.9023


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import evaluate

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Dataframe ko Hugging Face Dataset format me convert karna
def prepare_hf_data(df):
    return Dataset.from_pandas(df[['Text', 'label']].rename(columns={'Text': 'text'}))

hf_train = prepare_hf_data(train_df)
hf_val = prepare_hf_data(val_df)
hf_test = prepare_hf_data(test_df)

# Tokenize function
def tokenize_batch(batch):
    return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=128)

hf_train = hf_train.map(tokenize_batch, batched=True)
hf_val = hf_val.map(tokenize_batch, batched=True)
hf_test = hf_test.map(tokenize_batch, batched=True)

# Pre-trained BERT Model with 3 classes
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3).to(device)

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./bert_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    compute_metrics=compute_metrics
)

print("Fine-tuning BERT...")
trainer.train()

Map:   0%|          | 0/10500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Map:   0%|          | 0/2250 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fine-tuning BERT...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.609326,0.582564,0.747556,0.743366
2,0.427877,0.591927,0.757333,0.757070


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1314, training_loss=0.5641394343731791, metrics={'train_runtime': 505.6364, 'train_samples_per_second': 41.532, 'train_steps_per_second': 2.599, 'total_flos': 1381345443072000.0, 'train_loss': 0.5641394343731791, 'epoch': 2.0})

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 1. LSTM Evaluation
lstm_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for texts, labels in test_loader_lstm:
        texts = texts.to(device)
        outputs = lstm_model(texts)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

lstm_acc = accuracy_score(all_labels, all_preds)
lstm_f1 = f1_score(all_labels, all_preds, average='macro')

# 2. BERT Evaluation
bert_preds_raw = trainer.predict(hf_test)
bert_preds = np.argmax(bert_preds_raw.predictions, axis=1)
bert_labels = hf_test['label']

bert_acc = accuracy_score(bert_labels, bert_preds)
bert_f1 = f1_score(bert_labels, bert_preds, average='macro')

# Result Table
results = pd.DataFrame({
    "Model": ["Bi-LSTM Baseline", "Fine-Tuned BERT"],
    "Test Accuracy": [f"{lstm_acc * 100:.2f}%", f"{bert_acc * 100:.2f}%"],
    "Macro F1-Score": [f"{lstm_f1:.4f}", f"{bert_f1:.4f}"]
})

print("=== FINAL MODEL COMPARISON ===")
print(results.to_string(index=False))

=== FINAL MODEL COMPARISON ===
           Model Test Accuracy Macro F1-Score
Bi-LSTM Baseline        53.87%         0.5357
 Fine-Tuned BERT        75.47%         0.7550
